# 01. Binance Live Crypto Data Exploration
This notebook demonstrates how to load credentials from `.env`, query public/private Binance endpoints using `BinanceClient`, fetch top 24h gainers/volume pairs, and format market data for exploratory analysis.

In [1]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import plotly.express as px
from backend.binance_client import binance_client
from backend.config import API_KEY

print(f"Binance API Key Configured: {'Yes' if API_KEY else 'No'}")

Binance API Key Configured: Yes


## 1. Fetch Top 24h USDT Volume Leaders

In [2]:
usdt_tickers = binance_client.get_usdt_tickers()
df_tickers = pd.DataFrame(usdt_tickers)

# Clean numeric fields
numeric_cols = ['lastPrice', 'priceChangePercent', 'quoteVolume', 'volume', 'highPrice', 'lowPrice']
for col in numeric_cols:
    df_tickers[col] = df_tickers[col].astype(float)

# Sort by 24h volume
df_top_vol = df_tickers.sort_values(by='quoteVolume', ascending=False).head(20)
df_top_vol[['symbol', 'lastPrice', 'priceChangePercent', 'quoteVolume']].head(10)

,symbol,lastPrice,priceChangePercent,quoteVolume
11,USDCUSDT,1.00046,-0.021,1.884253e+09
0,BTCUSDT,66540.00000,1.740,1.242408e+09
1,ETHUSDT,1924.81000,1.385,6.383509e+08
222,USD1USDT,0.99979,-0.015,1.179113e+08
41,SOLUSDT,78.00000,0.554,1.163219e+08
270,OPNUSDT,0.07250,16.186,1.092888e+08
260,BANKUSDT,0.16710,-37.157,9.743852e+07
5,XRPUSDT,1.15100,3.572,8.824948e+07
99,DEXEUSDT,9.81600,-71.740,8.701422e+07
16,ZECUSDT,537.62000,-2.060,7.475751e+07


## 2. Visualize 24h Volume Distribution & Top Gainers

In [3]:
fig = px.bar(
    df_top_vol.head(15),
    x='symbol',
    y='quoteVolume',
    color='priceChangePercent',
    color_continuous_scale='RdYlGn',
    title='Top 15 Binance USDT Pairs by 24h Quote Volume ($)',
    labels={'quoteVolume': '24h Volume ($)', 'priceChangePercent': '24h Change (%)'}
)
fig.show()

## 3. Fetch Historical Candlesticks (BTC/USDT)

In [4]:
from backend.indicators import enrich_klines_dataframe

raw_klines = binance_client.get_klines('BTCUSDT', interval='1h', limit=100)
df_btc = enrich_klines_dataframe(raw_klines)
df_btc[['open_time', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'macd']].tail(10)

,open_time,open,high,low,close,volume,rsi,macd
90,2026-07-21 07:00:00,65913.00,66245.64,65866.00,66186.86,1338.71722,61.104848,315.735813
91,2026-07-21 08:00:00,66186.86,66354.00,66129.19,66226.44,1032.71252,70.653116,344.361953
92,2026-07-21 09:00:00,66226.43,66350.00,66144.00,66291.99,472.96398,74.409394,368.094547
93,2026-07-21 10:00:00,66292.00,66420.65,66176.55,66180.00,1179.04438,68.117875,373.559975
94,2026-07-21 11:00:00,66180.00,66350.00,66147.82,66345.59,752.24871,72.775970,386.794370
95,2026-07-21 12:00:00,66345.59,66640.00,66255.73,66558.82,1002.49408,79.575580,409.765086
96,2026-07-21 13:00:00,66558.81,66810.97,66315.62,66786.00,1358.43796,80.500851,441.215019
97,2026-07-21 14:00:00,66786.00,66956.15,66648.27,66842.30,1722.33044,78.283239,465.318356
98,2026-07-21 15:00:00,66842.29,66842.30,66517.52,66676.54,948.73464,84.539053,465.676944
99,2026-07-21 16:00:00,66676.53,66764.00,66539.62,66573.33,353.03500,78.827619,452.417753


In [6]:
df_btc

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,ema_9,ema_21,ema_50,ema_200,bb_middle,bb_upper,bb_lower,bb_bandwidth,atr,rvol
0,2026-07-17 13:00:00,63099.75,63180.00,62537.56,63153.99,1381.50027,2026-07-17 13:59:59.999,8.676115e+07,306096,574.56636000,...,63153.990000,63153.990000,63153.990000,63153.990000,NaN,NaN,NaN,NaN,642.440000,NaN
1,2026-07-17 14:00:00,63154.00,63518.00,63051.77,63174.00,1199.38162,2026-07-17 14:59:59.999,7.588861e+07,289884,517.61777000,...,63157.992000,63155.809091,63154.774706,63154.189104,NaN,NaN,NaN,NaN,629.853571,NaN
2,2026-07-17 15:00:00,63174.01,63452.00,62856.16,63452.00,702.84059,2026-07-17 15:59:59.999,4.437439e+07,193926,352.89734000,...,63216.793600,63182.735537,63166.430600,63157.152397,NaN,NaN,NaN,NaN,627.424031,NaN
3,2026-07-17 16:00:00,63452.00,63706.75,63312.01,63594.00,881.75053,2026-07-17 16:59:59.999,5.598169e+07,163102,469.94191000,...,63292.234880,63220.123216,63183.198027,63161.499139,NaN,NaN,NaN,NaN,610.803743,NaN
4,2026-07-17 17:00:00,63594.00,64228.27,63594.00,64049.86,1210.18889,2026-07-17 17:59:59.999,7.745698e+07,231758,571.88450000,...,63443.759904,63295.553832,63217.184771,63170.338551,NaN,NaN,NaN,NaN,612.479904,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026-07-21 12:00:00,66345.59,66640.00,66255.73,66558.82,1002.49408,2026-07-21 12:59:59.999,6.661802e+07,154107,626.01214000,...,66131.038914,65724.289727,65196.890163,64170.772290,65689.0855,66605.108114,64773.062886,2.788964,335.425865,1.445233
96,2026-07-21 13:00:00,66558.81,66810.97,66315.62,66786.00,1358.43796,2026-07-21 13:59:59.999,9.041893e+07,258062,734.51135000,...,66262.031131,65820.808843,65259.208195,64196.794456,65748.9865,66785.879561,64712.093439,3.154096,346.849018,1.866226
97,2026-07-21 14:00:00,66786.00,66956.15,66648.27,66842.30,1722.33044,2026-07-21 14:59:59.999,1.150958e+08,255021,949.00186000,...,66378.084905,65913.671675,65321.290227,64223.117894,65829.1010,66945.001388,64713.200612,3.390295,344.065516,2.212049
98,2026-07-21 15:00:00,66842.29,66842.30,66517.52,66676.54,948.73464,2026-07-21 15:59:59.999,6.325909e+07,191665,395.80559000,...,66437.775924,65983.023341,65374.437277,64247.530054,65905.8280,67033.766444,64777.889556,3.422879,342.687979,1.211292
